# Тестовое задание, атака на энергетический классификатор данных вне распределения

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.models import ResNet18_Weights, resnet18
import torchvision.transforms as transforms
import numpy as np
import random
from datetime import datetime

In [2]:
log_file = 'metrics.txt'

In [3]:
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True

set_seed()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'устройство: {device}')

устройство: cpu


In [4]:
transform = transforms.Compose([transforms.ToTensor()])
train_dtset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dtset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

def get_indices(dtset, is_id):
  targets = dtset.targets
  if is_id:
    return (targets >= 4).nonzero(as_tuple=True)[0]
  else:
    return (targets < 4).nonzero(as_tuple=True)[0]

id_train_idx = get_indices(train_dtset, is_id=True)
ood_train_idx = get_indices(train_dtset,is_id=False)
id_test_idx = get_indices(test_dtset, is_id=True)
ood_test_idx = get_indices(test_dtset, is_id=False)

id_train_dtset = torch.utils.data.Subset(train_dtset, id_train_idx)
ood_train_dtset = torch.utils.data.Subset(train_dtset, ood_train_idx)
id_test_dtset = torch.utils.data.Subset(test_dtset, id_test_idx)
ood_test_dtset = torch.utils.data.Subset(test_dtset, ood_test_idx)

batch = 128
id_train_loader = torch.utils.data.DataLoader(id_train_dtset, batch_size=batch, shuffle=True)
id_test_loader = torch.utils.data.DataLoader(id_test_dtset, batch_size=batch, shuffle=False)
ood_train_loader = torch.utils.data.DataLoader(ood_train_dtset, batch_size=batch, shuffle=False)



100%|██████████| 9.91M/9.91M [00:01<00:00, 5.73MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 151kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.41MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.7MB/s]


In [5]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for i in model.parameters():
  i.requires_grad = False

model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding = (3, 3), bias=False)
model.fc = nn.Linear(model.fc.in_features, 6)

model = model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 330MB/s]


In [6]:
crtr = nn.CrossEntropyLoss()
opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

epochs = 3
for epoch in range(epochs):
  model.train()
  running_loss = 0.0
  for inputs, labels, in id_train_loader:
    inputs = inputs.to(device)
    labels = (labels - 4).to(device)

    opt.zero_grad()
    outputs = model(inputs)
    loss = crtr(outputs, labels)
    loss.backward()
    opt.step()
    running_loss += loss.item()
  print(f'Эпоха {epoch + 1}/{epochs}, Потери: {running_loss/len(id_train_loader):.4f}')


Эпоха 1/3, Потери: 0.9247
Эпоха 2/3, Потери: 0.4799
Эпоха 3/3, Потери: 0.3728


In [7]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for inputs, labels in id_test_loader:
    inputs = inputs.to(device)
    labels = (labels - 4).to(device)

    outputs = model(inputs)
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
id_test_acc = 100 * correct / total
print(f'точность на ID тестовой выборке: {id_test_acc:.2f}')
f = open(log_file, 'a')
f.write(f'точность на ID тетсовой выборке: {id_test_acc:.2f}, время {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
f.close()

точность на ID тестовой выборке: 88.79


In [8]:
def compute_energy(logits, T=1.0):
  return -T * torch.logsumexp(logits / T, dim=1)

def get_energies(dataset):
  loader = torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)
  energies = []
  with torch.no_grad():
    for inputs, _ in loader:
      inputs = inputs.to(device)
      logits = model(inputs)
      e = compute_energy(logits)
      energies += e.tolist()
  return np.array(energies)

id_train_energies = get_energies(id_train_dtset)
ood_train_energies = get_energies(ood_train_dtset)


y_true_train = np.concatenate([np.ones(len(id_train_energies)), np.zeros(len(ood_train_energies))])
all_train_energies = np.concatenate([id_train_energies, ood_train_energies])

res_acc = 0
res_treshold = 0
tresholds = np.linspace(all_train_energies.min(), all_train_energies.max(), 1000)

for i in tresholds:
  y_pred = (all_train_energies < i).astype(int)
  acc = np.mean(y_pred == y_true_train)
  if acc > res_acc:
    res_acc = acc
    res_treshold = i

print(f'подобранный порог отказа: {res_treshold:.4f}')
f = open(log_file, 'a')
f.write(f'подобранный порог отказа: {res_treshold:.4f}, время {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
f.close()

подобранный порог отказа: -2.5222


In [9]:

sub_id_tst_dtst = torch.utils.data.Subset(id_test_dtset, range(500))
sub_ood_tst_dtst = torch.utils.data.Subset(ood_test_dtset, range(500))


id_tst_enrg = get_energies(sub_id_tst_dtst)
ood_tst_enrg = get_energies(sub_ood_tst_dtst)


y_ttest = np.concatenate([np.ones(500), np.zeros(500)])
all_etst = np.concatenate([id_tst_enrg, ood_tst_enrg])

y_ptest = (all_etst < res_treshold).astype(int)
btestacc = 100 * np.mean(y_ttest == y_ptest)
print(f'точность детектора: {btestacc}')
f = open(log_file, 'a')
f.write(f'точность детектора: {btestacc}, время {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
f.close()

точность детектора: 72.2


In [10]:
def pgd_attack(model, images, epsilon=8/255, alpha=1/255, iters=20):

  images = images.to(device)
  orig_img = images.clone()

  for _ in range(iters):
    images.requires_grad = True
    logits = model(images)
    loss = compute_energy(logits).sum()
    model.zero_grad()
    loss.backward()
    with torch.no_grad():
      adv_img = images - alpha * images.grad.sign()
      eta = torch.clamp(adv_img - orig_img, min=-epsilon, max=epsilon)
      images = torch.clamp(orig_img + eta, min=0.0, max=1.0)
  return images
print("применение атаки...")
adv_ood_enrg = []
ood_tst_loader = torch.utils.data.DataLoader(sub_ood_tst_dtst, batch_size=100, shuffle=False)
for inputs, _ in ood_tst_loader:
  adv_inputs = pgd_attack(model, inputs)
  with torch.no_grad():
    logits = model(adv_inputs)
    e = compute_energy(logits)
    adv_ood_enrg += e.tolist()
adv_ood_enrg = np.array(adv_ood_enrg)


применение атаки...


In [11]:
adv_tst_enrg = np.concatenate([id_tst_enrg, adv_ood_enrg])
pred_adv = (adv_tst_enrg < res_treshold).astype(int)
badvacc = 100 * np.mean(pred_adv == y_ttest)
adv_deg = btestacc - badvacc
print(f"точность детектора после атаки: {badvacc}")
print(f"деградация точности после атаки: {adv_deg}")

f = open(log_file, 'a')
f.write(f'точность детектора после атаки: {badvacc}, время {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
f.write(f'деградация точности детектора: {adv_deg}, время {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
f.close()

точность детектора после атаки: 40.1
деградация точности после атаки: 32.1


In [12]:
model_path = 'resnet18_ch.pth'
torch.save(model.state_dict(), model_path)
print(f'модель сохранена в {model_path}')

модель сохранена в resnet18_ch.pth
